# Leaf Disease Detection Using Image Segmentation & Feature Extraction
### College Mini-Project • IPA Group 12

This notebook demonstrates the step-by-step pipeline:
1. **Preprocessing**: Image resizing, Gaussian Blur noise reduction, and RGB $\rightarrow$ Gray $\rightarrow$ HSV color space conversions.
2. **Segmentation**: Otsu background isolation and HSV color thresholding + K-means clustering ($k=3$) for diseased spot extraction.
3. **Feature Extraction**: Color (HSV statistics), Texture (GLCM contrast, homogeneity, energy, correlation), and Shape metrics.
4. **Classification**: SVM training, model evaluation metrics, and confusion matrix visualization.

In [ ]:
import os
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add src directory to path
sys.path.append('../src')

from preprocess import load_and_preprocess_image
from segment import isolate_leaf_otsu, segment_diseased_region
from features import extract_all_features
from classify import train_and_evaluate
from pipeline import run_pipeline

## Step 1: Preprocessing Demonstration

In [ ]:
# Pick a sample image from dataset
sample_path = list(Path('../data/subset').rglob('*.jpg'))[0]
print(f"Processing sample image: {sample_path.name}")

prep = load_and_preprocess_image(str(sample_path))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(prep['original_rgb']); axes[0].set_title("Original RGB")
axes[1].imshow(prep['resized_rgb']); axes[1].set_title("Resized (256x256)")
axes[2].imshow(prep['blurred_rgb']); axes[2].set_title("Gaussian Blurred")
axes[3].imshow(prep['hsv']); axes[3].set_title("HSV Color Space")
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## Step 2: Image Segmentation (Otsu + HSV K-Means)

In [ ]:
leaf_mask = isolate_leaf_otsu(prep['blurred_rgb'])
seg = segment_diseased_region(prep['blurred_rgb'], prep['hsv'], leaf_mask)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(prep['blurred_rgb']); axes[0].set_title("Input Image")
axes[1].imshow(seg['leaf_mask'], cmap='gray'); axes[1].set_title("Otsu Leaf Mask")
axes[2].imshow(seg['diseased_mask'], cmap='hot'); axes[2].set_title("Diseased Spot Mask")
axes[3].imshow(seg['highlighted_overlay']); axes[3].set_title("Highlighted Contours")
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## Step 3: Feature Extraction (Color, GLCM Texture, Shape)

In [ ]:
feats, vec, names = extract_all_features(
    prep['blurred_rgb'], prep['hsv'], prep['gray'],
    seg['diseased_mask'], seg['leaf_mask']
)

print("Extracted Feature Dictionary:")
for k, v in sorted(feats.items()):
    print(f"  {k:22s}: {v:.4f}")

## Step 4: Model Training & Evaluation

In [ ]:
model_pack, accuracy = train_and_evaluate()
print(f"Final SVM Accuracy: {accuracy * 100:.2f}%")